In [ ]:
# Fixed settings for the local comparison. Paper-scale runs are not enabled here.
BATCH_SIZE = 128
NUM_EPOCHS = 20
DATA_SEED = 20261014
TRAIN_SEEDS = [7, 17, 27, 37, 47]
NUM_GLOBAL_VIEWS = 2
NUM_LOCAL_VIEWS = 4

# 02_lejepa_from_scratch.ipynb — LeJEPA objectives

**Goal**  
Compare this notebook's reduced SIGReg sketch with the Epps–Pulley SIGReg objective in LeJEPA.

**What you will measure**  
Five training seeds, a frozen CIFAR-10 linear probe, and feature spread for each method.

**Paper result**  
LeJEPA reports broad results across datasets and architectures. The paper's abstract reports 79% frozen linear-evaluation accuracy on ImageNet-1K with ViT-H/14. That is a paper result; this small CIFAR-10 study does not reproduce it.

**Method difference**  
The teaching arm uses 128 projections, `lambda=0.1`, and regularizes the concatenated views. The paper-aligned arm uses 512 projections, `lambda=0.05`, and averages SIGReg over views. The paper recommends 1,024 projections, two global plus eight local views, and batch size at least 128. This local study uses four local views, a small CNN, CIFAR-10, and 20 epochs.

**Data and compute**  
The notebook downloads the official 163 MB CIFAR-10 archive if it is not cached. It uses 640 unlabeled training images, 1,000 disjoint labeled probe images, and 1,000 held-out test images. CPU use is supported. The full study has ten model runs.

**Sources**  
- [LeJEPA paper, version 3](https://arxiv.org/html/2511.08544)  
- [LeJEPA official code](https://github.com/galilai-group/lejepa)  
- [CIFAR-10 official data page](https://www.cs.toronto.edu/~kriz/cifar.html)

## 1. Setup and imports

In [ ]:
import json
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CIFAR10

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial import embedding_stats, linear_probe_accuracy
from jepa_tutorial.storage import DATA_DIR
from jepa_tutorial.paper_losses import epps_pulley_sigreg_loss

# CPU keeps the recorded run portable and repeatable.
device = torch.device("cpu")
torch.set_num_threads(min(8, torch.get_num_threads()))
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "device": str(device),
    "torch_threads": torch.get_num_threads(),
})

In [ ]:
# Small deterministic checks for the paper-aligned objective.
probe = torch.randn(32, 16, generator=torch.Generator().manual_seed(101), requires_grad=True)
loss_a = epps_pulley_sigreg_loss(
    probe, num_projections=16, num_points=17,
    generator=torch.Generator().manual_seed(202),
)
loss_b = epps_pulley_sigreg_loss(
    probe, num_projections=16, num_points=17,
    generator=torch.Generator().manual_seed(202),
)
assert loss_a.ndim == 0 and torch.isfinite(loss_a)
assert torch.equal(loss_a, loss_b), "SIGReg must repeat with the same generator seed"
loss_a.backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
try:
    epps_pulley_sigreg_loss(torch.zeros(2, 3, 4))
except ValueError:
    pass
else:
    raise AssertionError("SIGReg must reject an input that is not a matrix")
print("SIGReg shape, finiteness, gradient, and repeatability checks passed")


## 2. Data: fixed, disjoint CIFAR-10 subsets

Labels are used only to make balanced subsets and fit the probe. The pretraining loader returns images only. We do not add downloaded data to Git.

In [ ]:
DATA_ROOT = DATA_DIR
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2430, 0.2610)
normalize = T.Normalize(MEAN, STD)
global_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    normalize,
])
local_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 0.6)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    normalize,
])
eval_transform = T.Compose([T.ToTensor(), normalize])

try:
    train_base = CIFAR10(root=str(DATA_ROOT), train=True, download=True)
    test_base = CIFAR10(root=str(DATA_ROOT), train=False, download=True)
except Exception as exc:
    raise RuntimeError(
        "Could not load CIFAR-10. See https://www.cs.toronto.edu/~kriz/cifar.html"
    ) from exc


def balanced_indices(labels, candidates, per_class, rng):
    labels = np.asarray(labels)
    candidates = np.asarray(candidates, dtype=np.int64)
    selected = []
    for class_id in range(10):
        pool = candidates[labels[candidates] == class_id]
        if len(pool) < per_class:
            raise ValueError(f"Not enough CIFAR-10 examples for class {class_id}")
        selected.extend(rng.choice(pool, per_class, replace=False).tolist())
    return np.asarray(sorted(selected), dtype=np.int64)


split_rng = np.random.default_rng(DATA_SEED)
all_train_indices = np.arange(len(train_base))
ssl_indices = balanced_indices(train_base.targets, all_train_indices, 64, split_rng)
probe_candidates = np.setdiff1d(all_train_indices, ssl_indices)
probe_indices = balanced_indices(train_base.targets, probe_candidates, 100, split_rng)
test_indices = balanced_indices(
    test_base.targets, np.arange(len(test_base)), 100, split_rng
)


class MultiViewSubset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, _ = self.dataset[int(self.indices[position])]
        views = [global_transform(image) for _ in range(NUM_GLOBAL_VIEWS)]
        views.extend(local_transform(image) for _ in range(NUM_LOCAL_VIEWS))
        return views


class LabeledSubset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, label = self.dataset[int(self.indices[position])]
        return eval_transform(image), int(label)


probe_loader = DataLoader(
    LabeledSubset(train_base, probe_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    LabeledSubset(test_base, test_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
print({"ssl_images": len(ssl_indices), "probe_images": len(probe_indices), "test_images": len(test_indices), "data_seed": DATA_SEED})

## 3. Small encoder

The four-stage CNN has a 256-dimensional output and about 455,000 parameters. It has no pretrained weights. Both objectives use the same model.

In [ ]:
class SmallEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.embedding = nn.Linear(256, embed_dim)

    def forward(self, images):
        return self.embedding(self.features(images).flatten(1))


model_parameters = sum(p.numel() for p in SmallEncoder().parameters())
print({"backbone_parameters": model_parameters, "embedding_dim": 256})

## 4. Prediction and SIGReg

The prediction term matches each view to the mean of the two global views. The teaching arm matches only local views and regularizes their concatenation. The paper-aligned arm matches all views and averages one SIGReg calculation per view.

In [ ]:
def teaching_prediction_loss(view_embeddings):
    center = view_embeddings[:NUM_GLOBAL_VIEWS].mean(dim=0)
    return (view_embeddings[NUM_GLOBAL_VIEWS:] - center).square().mean()


def paper_prediction_loss(view_embeddings):
    center = view_embeddings[:NUM_GLOBAL_VIEWS].mean(dim=0)
    return (view_embeddings - center).square().mean()


def collect_features(model, loader):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            features.append(model(images.to(device)).cpu().numpy())
            labels.append(targets.numpy())
    return np.concatenate(features), np.concatenate(labels)


def run_method(method, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = SmallEncoder().to(device)
    train_generator = torch.Generator().manual_seed(seed + 1000)
    loss_generator = torch.Generator().manual_seed(seed + 2000)
    loader = DataLoader(
        MultiViewSubset(train_base, ssl_indices),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator,
        num_workers=0,
    )
    optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-5)
    started = time.perf_counter()
    final_loss = float("nan")

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_losses = []
        for batch_views in loader:
            batch_size = batch_views[0].shape[0]
            images = torch.cat([view.to(device) for view in batch_views], dim=0)
            embeddings = model(images)
            view_embeddings = embeddings.view(
                NUM_GLOBAL_VIEWS + NUM_LOCAL_VIEWS, batch_size, -1
            )
            if method == "teaching_sketch":
                pred = teaching_prediction_loss(view_embeddings)
                sig = epps_pulley_sigreg_loss(
                    embeddings,
                    num_projections=128,
                    num_points=17,
                    generator=loss_generator,
                )
                loss = 0.9 * pred + 0.1 * sig
            else:
                pred = paper_prediction_loss(view_embeddings)
                sig = torch.stack([
                    epps_pulley_sigreg_loss(
                        view,
                        num_projections=512,
                        num_points=17,
                        generator=loss_generator,
                    )
                    for view in view_embeddings
                ]).mean()
                loss = 0.95 * pred + 0.05 * sig
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch + 1}")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach()))
        final_loss = float(np.mean(epoch_losses))

    probe_x, probe_y = collect_features(model, probe_loader)
    test_x, test_y = collect_features(model, test_loader)
    accuracy = linear_probe_accuracy(probe_x, probe_y, test_x, test_y)
    stats = embedding_stats(test_x)
    return {
        "method": method,
        "seed": seed,
        "status": "completed",
        "probe_top1": accuracy,
        "embedding_mean_std": stats["mean_std"],
        "trace_cov": stats["trace_cov"],
        "mean_abs_offdiag_cov": stats["mean_abs_offdiag_cov"],
        "final_train_loss": final_loss,
        "elapsed_seconds": time.perf_counter() - started,
    }

## 5. Five-seed comparison

The methods share data indices, model, optimizer, batch size, and epoch count. A failed or non-finite run stays in the result table.

In [ ]:
METHODS = ["teaching_sketch", "paper_aligned_sigreg"]
RUNS = []
for method in METHODS:
    for seed in TRAIN_SEEDS:
        try:
            record = run_method(method, seed)
        except Exception as exc:
            record = {"method": method, "seed": seed, "status": "failed", "error": f"{type(exc).__name__}: {exc}"}
        RUNS.append(record)
        print({"method": method, "seed": seed, "status": record["status"], "probe_top1": record.get("probe_top1"), "elapsed_seconds": record.get("elapsed_seconds"), "error": record.get("error")})

This study does not save model checkpoints. The fixed data split, settings, and per-seed results are recorded in the report.

## 6. Metrics and limits

The frozen probe uses the fixed 1,000-image labeled train subset and 1,000-image held-out test subset. Feature statistics describe the same test images. A small interval across five seeds is uncertain.

In [ ]:
from scipy.stats import t as student_t


SUMMARY = {}
for method in METHODS:
    selected = [r for r in RUNS if r["method"] == method]
    SUMMARY[method] = {
        "runs": selected,
    }

# Keep the raw run rows and calculate aggregates by method.
for method in METHODS:
    selected = [r for r in RUNS if r["method"] == method and r["status"] == "completed"]
    for metric in ("probe_top1", "embedding_mean_std", "trace_cov", "mean_abs_offdiag_cov"):
        values = np.asarray([r[metric] for r in selected], dtype=float)
        if len(values) >= 2:
            half = student_t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))
            ci = [float(values.mean() - half), float(values.mean() + half)]
        else:
            ci = None
        SUMMARY[method][metric] = {"n": len(values), "mean": float(values.mean()) if len(values) else None, "median": float(np.median(values)) if len(values) else None, "ci95": ci, "interval": "two-sided Student-t; uncertain at n=5"}

paired_deltas = {}
for metric in ("probe_top1", "embedding_mean_std", "trace_cov", "mean_abs_offdiag_cov"):
    deltas = []
    for seed in TRAIN_SEEDS:
        by_method = {
            run["method"]: run for run in RUNS
            if run["seed"] == seed and run["status"] == "completed"
        }
        if all(method in by_method for method in METHODS):
            deltas.append(by_method["paper_aligned_sigreg"][metric] - by_method["teaching_sketch"][metric])
    values = np.asarray(deltas, dtype=float)
    half = student_t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values)) if len(values) >= 2 else None
    paired_deltas[metric] = {
        "n": len(values),
        "paper_aligned_minus_teaching_mean": float(values.mean()) if len(values) else None,
        "ci95": [float(values.mean() - half), float(values.mean() + half)] if half is not None else None,
        "interval": "paired two-sided Student-t; uncertain at n=5",
    }

print(json.dumps({"settings": {"data_seed": DATA_SEED, "training_seeds": TRAIN_SEEDS, "ssl_images": len(ssl_indices), "probe_images": len(probe_indices), "test_images": len(test_indices), "backbone_parameters": model_parameters, "batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS, "learning_rate": 0.003, "weight_decay": 1e-5, "global_views": NUM_GLOBAL_VIEWS, "local_views": NUM_LOCAL_VIEWS, "teaching_projections": 128, "paper_aligned_projections": 512, "teaching_lambda": 0.1, "paper_aligned_lambda": 0.05}, "runs": RUNS, "summary": SUMMARY, "paired_deltas": paired_deltas}, indent=2))

## Paper reproduction boundary

The paper's ImageNet-1K result is paper-reported. These CIFAR-10 measurements are local and use a small model, small subsets, fewer projections, fewer local views, and 20 epochs. They are a local objective study, not a reproduction of LeJEPA's benchmark results.